# CySent Live RL Training — Phi-3.5-mini-instruct

Real-time REINFORCE + LoRA training against the live CySent environment.
**Not** static dataset — the model observes, acts, and learns from live env rewards.

| Setting | Value |
|---------|-------|
| Model | `microsoft/Phi-3.5-mini-instruct` (3.8B, fp16) |
| RL algo | REINFORCE with baseline |
| LoRA r | 8 (auto-detects `qkv_proj` for Phi arch) |
| Episodes | 200 (max) |
| Max steps/ep | 40 |
| GPU | Colab T4 (free tier sufficient) |
| Est. time | 25-40 min |
| Est. cost | Free tier / ~$0.50 if paid |

In [ ]:
# Cell 1 — Install dependencies
!pip install -q --upgrade pip
!pip install -q "transformers>=4.44.2" "accelerate>=0.31.0" peft bitsandbytes
!pip install -q stable-baselines3 gymnasium numpy

# Clone CySent repo
import os
if not os.path.exists('/content/CySent'):
    !git clone https://github.com/sxchin-01/CySent.git /content/CySent

os.chdir('/content/CySent')
!pip install -q -r backend/requirements.txt
os.environ['PYTHONPATH'] = '/content/CySent'
print('Setup complete. If versions changed significantly, Runtime > Restart runtime once.')

In [ ]:
# Cell 2 — Live RL Training (Phi-3.5)
import sys, os
os.chdir('/content/CySent')
sys.path.insert(0, '/content/CySent')

from backend.train.train_hf_rl import train_hf_rl

summary = train_hf_rl(
    model_id='microsoft/Phi-3.5-mini-instruct',
    output_dir='outputs/phi35_rl_adapter',
    episodes=200,
    max_steps=40,
    lr=5e-5,
    gamma=0.97,
    temperature=0.7,
    lora_r=8,
    lora_alpha=16,
    seed=42,
    grad_accum=4,
    early_stop_reward=3.0,
    save_every=25,
)

import json
print('\n' + json.dumps(summary, indent=2))

In [ ]:
# Cell 3 — 4-way Benchmark: phi35_rl vs qwen_rl vs PPO vs random
from backend.train.train_hf_rl import benchmark_hf_adapter
import json

results = benchmark_hf_adapter(
    model_id='microsoft/Phi-3.5-mini-instruct',
    adapter_path='outputs/phi35_rl_adapter/final_adapter',
    episodes=10,
    max_steps=40,
    seed=42,
    extra_adapters=[
        ('qwen_rl', 'Qwen/Qwen2.5-3B-Instruct', 'outputs/hf_rl_adapter/final_adapter'),
    ],
)

print('\n=== Benchmark Results ===')
header = f"{'Agent':<24} {'Avg Reward':>12} {'Avg Risk':>10} {'Avg Breaches':>14} {'Avg Steps':>10}"
print(header)
print('-' * len(header))
for agent, stats in results.items():
    print(f"{agent:<24} {stats['avg_reward']:>+12.3f} {stats['avg_risk']:>10.3f} "
          f"{stats['avg_breaches']:>14.3f} {stats['avg_steps']:>10.1f}")

from pathlib import Path
out = Path('outputs/phi35_rl_adapter/benchmark_results.json')
out.write_text(json.dumps(results, indent=2))
print(f'\nSaved to {out}')

In [ ]:
# Cell 4 — Save adapter to Google Drive (optional)
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
dst = '/content/drive/MyDrive/cysent_phi35_rl_adapter'
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree('outputs/phi35_rl_adapter/final_adapter', dst)
print(f'Adapter copied to {dst}')

shutil.copy2('outputs/phi35_rl_adapter/benchmark_results.json',
             '/content/drive/MyDrive/cysent_phi35_benchmark.json')
print('Benchmark results copied to Drive.')

In [ ]:
# Cell 5 — Resume training from checkpoint (optional)
# Uncomment and adjust the checkpoint path if resuming
# To resume: load PeftModel from checkpoint, wrap in train_hf_rl manually,
# or re-run Cell 2 with output_dir pointing to existing checkpoint folder.
# The script auto-saves checkpoints every `save_every` episodes.

# Example: verify a checkpoint loads correctly
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

ckpt = 'outputs/phi35_rl_adapter/checkpoint_ep25'  # adjust as needed
base = AutoModelForCausalLM.from_pretrained(
    'microsoft/Phi-3.5-mini-instruct',
    torch_dtype=torch.float16, device_map='auto', trust_remote_code=True,
)
model = PeftModel.from_pretrained(base, ckpt)
print(f'Checkpoint loaded from {ckpt}: {sum(p.numel() for p in model.parameters()):,} params')